In [ ]:
# Phase 6 confirm: ONE config over all five folds. Runs on prep v2 artifacts (130mm physical-scale crop
# at 336px, six plane/weighting slots with a presence mask, contiguous 3-slice
# groups) and the report-hash folds v3.
#
# The screens ran fold 0 only. This produces a pooled out-of-fold prediction over
# every labelled study, which is what the promotion rule reads, plus a gold
# transfer from the five-fold mean. Members are ensembled across configs
# afterwards, outside this kernel.
import glob, os, shutil, sys, time

GIT_SHA = '801df73-wip'

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

import inspect
import torch

assert torch.cuda.is_available()
major, minor = torch.cuda.get_device_capability(0)
print(f'GPU: {torch.cuda.get_device_name(0)}, sm_{major}{minor}')
assert (major, minor) >= (7, 0), 'need T4, not P100'
device = 'cuda'

# A stale rsna-knee-src version cost 7.5 minutes of a Phase 5 screen before the
# run noticed it was training the wrong code (NOTES 2026-09-05). These asserts
# fail in the first cell instead.
from knee.dataset import PreppedSlotDataset
from knee.model import KneeModel
from knee.reports import report_group_key
from knee.train import differential_param_groups, make_folds, train_one_epoch
# A stale src dataset has now cost three runs. The old guard listed the previous
# change by hand, so it only ever caught yesterday's mistake -- this one is
# derived from what the configs below actually ask for, so a config needing new
# code cannot silently outrun the dataset version. Tuples, not sets: this cell is
# an f-string and braces here would be interpolated.
REQUIRED = [
    ('KneeModel.__init__', KneeModel.__init__, ('head', 'backbone_kwargs')),
    ('KneeModel.forward', KneeModel.forward, ('mask',)),
    ('make_folds', make_folds, ('groups',)),
    ('differential_param_groups', differential_param_groups, ('backbone_lr', 'head_lr')),
]
for label, fn, needed in REQUIRED:
    have = set(inspect.signature(fn).parameters)
    missing = [n for n in needed if n not in have]
    assert not missing, (label + ' is missing ' + repr(missing) +
                         ' -- rsna-knee-src is stale. Re-version it before pushing.')
assert hasattr(KneeModel, 'attention_weights'), 'rsna-knee-src predates the c2 head'
print('src carries every Phase 6 change this run depends on | GIT_SHA', GIT_SHA)

In [ ]:
# GATE -- the corpus prep census, read from the shard manifests rather than
# downloaded: 9.5GB of artifacts do not need to leave Kaggle to be checked.
import numpy as np
import pandas as pd

PREPPED_DIRS = sorted(glob.glob('/kaggle/input/**/prepped', recursive=True))
uid_to_npz = {}
for d in PREPPED_DIRS:
    for f in os.listdir(d):
        if f.endswith('.npz'):
            uid_to_npz[f[:-4]] = os.path.join(d, f)

manifest = pd.concat([pd.read_csv(f) for f in
                      sorted(glob.glob('/kaggle/input/**/prep_manifest_shard*.csv', recursive=True))])
failed = pd.concat([pd.read_csv(f) for f in
                    sorted(glob.glob('/kaggle/input/**/prep_failed_shard*.csv', recursive=True))])
series_meta = pd.concat([pd.read_csv(f) for f in
                         sorted(glob.glob('/kaggle/input/**/prep_series_meta_shard*.csv', recursive=True))])

# One root by symlink, as Phase 5 did: the loader takes a directory and the
# artifacts arrive in four mounted shards. Symlinks cost no space.
NPZ_ROOT = '/kaggle/working/prepped_all'
os.makedirs(NPZ_ROOT, exist_ok=True)
for uid, p in uid_to_npz.items():
    dst = os.path.join(NPZ_ROOT, f'{uid}.npz')
    if not os.path.exists(dst):
        os.symlink(p, dst)

print(f'{len(PREPPED_DIRS)} shards, {len(uid_to_npz)} artifacts, '
      f'{len(manifest)} manifest rows, {len(failed)} failed studies')
assert len(uid_to_npz) == len(manifest), 'artifact count and manifest disagree'
assert len(uid_to_npz) + len(failed) == 4407, 'shards do not cover the corpus'

mm = series_meta['mm_per_px'].unique()
assert len(mm) == 1, f'physical scale is not constant: {mm}'
print(f'PASS: {len(series_meta)} stored series, {series_meta.PixelSpacing.nunique()} distinct '
      f'PixelSpacing values, all at {mm[0]:.5f} mm/px')

from knee.dicom import SLOTS
slot_names = [n for n, _, _ in SLOTS]
fill = manifest.slots_filled.fillna('').str.get_dummies(sep='|')
print(f'\nmean slots/study {manifest.n_slots_filled.mean():.2f} of 6')
for name in slot_names:
    n = int(fill[name].sum()) if name in fill else 0
    print(f'  {name:12s} {n:5d}  {n / len(manifest):6.1%}')
print(f'\ndecode failures: {int(manifest.n_decode_failures.sum())}')
print(f'laterality routes: {dict(manifest.route.value_counts())}')

In [ ]:
# Folds v3 and labels. Folds are regenerated here rather than uploaded, and
# pinned by hash: the experiment discipline requires one frozen fold assignment
# across every Phase 6 run, and a silent drift would make every paired delta
# meaningless without failing anything.
import hashlib

from knee.infer import LABEL_COLUMNS
from knee.metrics import macro_auc, paired_macro_auc_delta, per_label_auc
from knee.train import Timer, evaluate, log_experiment, train_val_split
from torch.utils.data import DataLoader

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
all_uids = sorted(train_df['StudyInstanceUID'].astype(str))
assert len(all_uids) == 4407

groups = {u: report_group_key(r)
          for u, r in zip(train_df['StudyInstanceUID'].astype(str), train_df['Report'])}
folds = make_folds(all_uids, n_folds=5, seed=0, groups=groups)
FOLDS_PRIMARY_V3_SHA256 = 'f1d6ba7c341f8d2e82241cb4ddd5ca5131b0c6d2033eb6d2e43bcb597cf12868'
assert hashlib.sha256(
    '\n'.join(f'{u},{folds[u]}' for u in sorted(folds)).encode()
).hexdigest() == FOLDS_PRIMARY_V3_SHA256, 'fold assignment drifted from folds_primary_v3.csv'
n_straddle = sum(1 for _, g in pd.DataFrame(
    {'g': [groups[u] for u in all_uids], 'f': [folds[u] for u in all_uids]}
).groupby('g') if g.f.nunique() > 1)
print(f'folds v3 verified; {n_straddle} report groups straddle folds (v2 had 47)')

gold_cols = [c for c in train_df.columns if c not in ('StudyInstanceUID', 'Report')]
gold_df = train_df[train_df[gold_cols].notna().any(axis=1)]
holdout = frozenset(gold_df['StudyInstanceUID'].astype(str))
print(f'{len(holdout)} gold studies held out of both sides of every split')

# Step 3 re-baselines on the same label source Phase 5 trained on; swapping the
# label target is c0, a separate cell, so the two changes stay attributable.
pseudo = pd.read_csv(glob.glob('/kaggle/input/**/pseudo_labels_qwen3_4b.csv', recursive=True)[0])
labels_all = pseudo[['StudyInstanceUID'] + [f'score_{l}' for l in LABEL_COLUMNS]]
labels_all.columns = ['StudyInstanceUID'] + LABEL_COLUMNS

# Two frames, as Phase 5 used: soft scores are the training target, but AUC is
# scored against a binarized one -- roc_auc_score rejects a continuous y_true,
# and the soft target is a confidence, not a label.
labels_eval = labels_all.copy()
labels_eval[LABEL_COLUMNS] = (labels_all[LABEL_COLUMNS].to_numpy(dtype=float) >= 0.5).astype(float)
print(f'{len(labels_all)} pseudo-labelled studies (soft for training, binarized for scoring)')

# c0's alternative target. steven_v4 is the best single public source on the 58
# gold studies (0.8927 vs our 0.8616, paired delta +0.031 [+0.004, +0.058]).
#
# NOT the 5-source rank blend, which scored higher on gold (0.8976): percentile
# ranks are uniform by construction, so every label's target mean is exactly
# 0.500 and all prevalence information is destroyed -- Fracture would be trained
# at 0.5 against a real rate of 0.014. AUC never notices, because AUC reads
# order only. That is exactly why rank-averaging is right for combining
# predictions at submission time and wrong for building a target.
LABEL_SOURCES = {'ours_qwen3_4b': (labels_all, labels_eval)}
steven = glob.glob('/kaggle/input/**/llm_labels_v4_blend.csv', recursive=True)
if steven:
    alt = pd.read_csv(steven[0])
    alt['StudyInstanceUID'] = alt['StudyInstanceUID'].astype(str)
    alt = alt[['StudyInstanceUID'] + LABEL_COLUMNS]
    alt_eval = alt.copy()
    alt_eval[LABEL_COLUMNS] = (alt[LABEL_COLUMNS].to_numpy(dtype=float) >= 0.5).astype(float)
    LABEL_SOURCES['steven_v4'] = (alt, alt_eval)
    rate = alt[LABEL_COLUMNS].mean()
    print(f'steven_v4 loaded: {len(alt)} studies, positive rate '
          f'{rate.min():.3f}-{rate.max():.3f} (ours: '
          f'{labels_all[LABEL_COLUMNS].mean().min():.3f}-'
          f'{labels_all[LABEL_COLUMNS].mean().max():.3f})')
print('label sources available:', list(LABEL_SOURCES))

# The arbiter for c0. Scoring each model against its OWN label source would be
# circular -- that measures how learnable a label set is, not which one trains a
# better model, and each source would be graded by its own marker. The 58 gold
# studies are rubric-graded from images, excluded from every training split, and
# identical for both arms, so they are the only unbiased comparison available
# when the target itself is the variable. n=58 is thin (MCL has 9 positives) and
# the per-label numbers are directional only; the macro is the read.
gold_labels = gold_df[['StudyInstanceUID'] + LABEL_COLUMNS].copy()
gold_labels['StudyInstanceUID'] = gold_labels['StudyInstanceUID'].astype(str)
gold_uids = [u for u in gold_labels['StudyInstanceUID'] if u in uid_to_npz]
print(f'gold arbiter: {len(gold_uids)} of 58 studies have a prepped artifact')

In [ ]:
# One config, five folds. Each fold's model predicts the held-out fold and all 58
# gold studies; gold is excluded from training in every fold, so the five gold
# prediction sets average into an honest transfer number for the fold ensemble.
CONFIG = ('c2_6slot_attn', 'steven_v4', slot_names, 5, 224, 8, 8, 'slot_attention',
          'efficientnet_b0', None)
HYPOTHESIS = ('Phase 6 confirm of the screened config over all 5 folds of '
              'primary_v3, target steven_v4. Pooled OOF is the promotion metric; '
              'gold transfer is the tier that resembles the hidden test set.')

(name, source, cfg_slots, n_groups, out_size, epochs, batch, head,
 backbone, lrs) = CONFIG
train_labels, eval_labels = LABEL_SOURCES[source]

EXPERIMENTS_CSV = '/kaggle/working/experiments.csv'
_HEADER = ('date,git_sha,config_hash,hypothesis,fold_set,seed,acl_auc,mcl_auc,'
           'medial_meniscus_auc,lateral_meniscus_auc,medial_oa_auc,lateral_oa_auc,'
           'pf_oa_auc,effusion_auc,synovitis_auc,bakers_auc,contusion_auc,fracture_auc,'
           'macro_auc,paired_delta,train_minutes,inference_seconds,promoted')
with open(EXPERIMENTS_CSV, 'w') as f:
    print(_HEADER, file=f)

def loader(uids, labels_df, shuffle=False, augment=False):
    ds = PreppedSlotDataset(uids, NPZ_ROOT, labels_df=labels_df, n_groups=n_groups,
                            out_size=out_size, slots=cfg_slots, augment=augment)
    return DataLoader(ds, batch_size=batch, shuffle=shuffle, num_workers=2)

labeled_uids = [u for u in all_uids if u not in holdout and u in uid_to_npz]
row_of = {u: i for i, u in enumerate(labeled_uids)}
oof_true = np.full((len(labeled_uids), len(LABEL_COLUMNS)), np.nan)
oof_pred = np.full_like(oof_true, np.nan)
gold_by_fold = []
print(f'{name}: {len(labeled_uids)} labelled studies across 5 folds')

t_start = time.time()
for fold in range(5):
    tr_uids, va_uids = train_val_split(folds, val_fold=fold, exclude_uids=holdout)
    tr_uids = [u for u in tr_uids if u in uid_to_npz]
    va_uids = [u for u in va_uids if u in uid_to_npz]

    train_loader = loader(tr_uids, train_labels, shuffle=True, augment=True)
    val_loader = loader(va_uids, eval_labels)
    extra = {'img_size': out_size} if backbone.startswith('vit_') else {}
    model = KneeModel(backbone_name=backbone, num_labels=12, pretrained=True,
                      head=head, **extra).to(device)
    if lrs is None:
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
        max_lr = 3e-4
    else:
        optimizer = torch.optim.AdamW(differential_param_groups(model, *lrs),
                                      weight_decay=0.02)
        max_lr = [lrs[0] * 3, lrs[1] * 3]
    scaler = torch.amp.GradScaler('cuda')
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=max_lr, epochs=epochs, steps_per_epoch=len(train_loader))

    t0 = time.time()
    for epoch in range(epochs):
        train_one_epoch(model, train_loader, optimizer, device=device,
                        scaler=scaler, scheduler=scheduler)
    minutes = (time.time() - t0) / 60

    y_true, y_pred = evaluate(model, val_loader, device=device)
    for uid, yt, yp in zip(va_uids, y_true, y_pred):
        oof_true[row_of[uid]] = yt
        oof_pred[row_of[uid]] = yp
    aucs = per_label_auc(y_true, y_pred)
    fold_macro = macro_auc(y_true, y_pred)

    g_true, g_pred = evaluate(model, loader(gold_uids, gold_labels), device=device)
    gold_by_fold.append(g_pred)
    print(f'fold {fold}: val macro {fold_macro:.4f}, gold {macro_auc(g_true, g_pred):.4f}, '
          f'{minutes:.1f} min ({(time.time() - t_start) / 3600:.1f}h elapsed)')

    log_experiment(EXPERIMENTS_CSV, git_sha=GIT_SHA, config_hash=name,
                   hypothesis=HYPOTHESIS, fold_set=f'primary_v3_fold{fold}', seed=fold,
                   per_label_auc={l: float(a) for l, a in zip(LABEL_COLUMNS, aucs)},
                   macro_auc=float(fold_macro), paired_delta=float('nan'),
                   train_minutes=minutes, inference_seconds=float('nan'), promoted=False)
    torch.save(model.state_dict(), f'/kaggle/working/{name}_fold{fold}.pt')
    del model, optimizer, scaler, scheduler
    torch.cuda.empty_cache()

assert not np.isnan(oof_pred).any(), 'a study never landed in a validation fold'
np.save(f'/kaggle/working/oof_pred_{name}.npy', oof_pred)
np.save(f'/kaggle/working/oof_true_{name}.npy', oof_true)
np.save(f'/kaggle/working/gold_folds_{name}.npy', np.stack(gold_by_fold))
np.save(f'/kaggle/working/gold_true_{name}.npy', g_true)
pd.DataFrame({'StudyInstanceUID': labeled_uids}).to_csv(
    f'/kaggle/working/oof_uids_{name}.csv', index=False)

In [ ]:
# The two tiers, and the honest framing of each.
pooled = macro_auc(oof_true, oof_pred)
gold_mean = np.mean(gold_by_fold, axis=0)   # the fold ensemble, as it will be submitted
gold_macro = macro_auc(g_true, gold_mean)

print(f'pooled OOF macro over {len(labeled_uids)} studies: {pooled:.4f}')
print(f'gold transfer, 5-fold mean (n=58):                {gold_macro:.4f}')
print(f'  Phase 5 for reference: pooled OOF 0.8523 on folds v2 against our own '
      f'pseudo-labels, gold 0.8189. NOT a paired comparison -- different folds, '
      f'different target, different pixels.')
print()
print(f'{"label":20s} {"OOF":>8s} {"gold":>8s}')
for l, a, b in zip(LABEL_COLUMNS, per_label_auc(oof_true, oof_pred),
                   per_label_auc(g_true, gold_mean)):
    print(f'{l:20s} {a:8.4f} {b:8.4f}')

log_experiment(EXPERIMENTS_CSV, git_sha=GIT_SHA, config_hash=name + '_pooled',
               hypothesis=HYPOTHESIS, fold_set='primary_v3_pooled', seed=0,
               per_label_auc={l: float(a) for l, a in
                              zip(LABEL_COLUMNS, per_label_auc(oof_true, oof_pred))},
               macro_auc=float(pooled), paired_delta=float('nan'),
               train_minutes=(time.time() - t_start) / 60,
               inference_seconds=float('nan'), promoted=False)
print(f'\ntotal {(time.time() - t_start) / 3600:.2f}h')